# TetraFT — heal_kl_50m + polish_kl_5m

**Attach datasets**
- `tetraft-code` — flat `.py` + this notebook (**refresh** after code changes)
- `tetraft-fineweb-edu-50m` — `train.jsonl`, `val.jsonl`
- **Session B:** Dataset with Session A `checkpoint-final` (**full** = opt+sched)
- **Session P (polish):** Dataset with Session B `checkpoint-final` (**weights-only OK**)

| Setting | Value |
|---------|--------|
| Accelerator | **GPU** |
| Internet | **ON** first run |

| SESSION | Preset | Steps | Resume | Notes |
|---------|--------|------:|--------|-------|
| **A** | `heal_kl_50m` | 0→6104 | none | full opt ckpt |
| **B** | `heal_kl_50m` | 6104→12207 | A full | → SOTA ~34.38 |
| **P** | `polish_kl_5m` | 12207→13487 | B weights | lr **2e-5** constant; gate **&lt;34.38** |

**Polish DNA:** skip GDN, α=0.5/T=2/β=0.01, cosine `min_lr_ratio=1.0`, `warmup=0`, fresh Adam (weights-only).  
**Critical:** `max_steps` must be **&gt; resumed_step** or train no-ops.

Logic in `run_smoke.py` — notebook is glue only.


In [ ]:
# Qwen3.5 needs recent transformers (model_type qwen3_5).
# If KeyError qwen3_5:
# %pip install -U "git+https://github.com/huggingface/transformers.git"
%pip install -q -U transformers accelerate bitsandbytes sentencepiece

In [ ]:
import sys
from pathlib import Path

def find_file(name: str) -> Path:
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]
    for root in roots:
        if not root.exists():
            continue
        direct = root / name
        if direct.is_file():
            return direct
        for p in root.rglob(name):
            if p.is_file():
                return p
    raise FileNotFoundError(name)

code_py = find_file("run_smoke.py")
code_root = code_py.parent
sys.path.insert(0, str(code_root))
print("code:", code_root)

train_path = find_file("train.jsonl")
val_path = find_file("val.jsonl")
print("train:", train_path)
print("val:", val_path)

import transformers
print("transformers", transformers.__version__)

from config import SMOKE_PRESETS
assert "heal_kl_50m" in SMOKE_PRESETS, "heal_kl_50m missing — refresh tetraft-code"
assert "polish_kl_5m" in SMOKE_PRESETS, "polish_kl_5m missing — refresh tetraft-code"
assert "scout_kl_5m" in SMOKE_PRESETS
print("presets ok:", sorted(SMOKE_PRESETS))

In [ ]:
from run_smoke import run_smoke
import argparse
import shutil
from pathlib import Path

# =============================================================================
# SESSION: "A" | "B" | "P" (polish from B weights @ lr 2e-5)
# =============================================================================
SESSION = "P"  # <-- A | B | P

CLEAR_OUTPUT = True

if SESSION.upper() == "A":
    PRESET = "heal_kl_50m"
    MAX_STEPS = 6104
    SAVE_OPTIMIZER = True
    RESUME = None
    SKIP_SHOCK = False
    SKIP_ORIG = False
    LEARNING_RATE = None
    LR_SCHEDULER = None
    MIN_LR_RATIO = None
    WARMUP_STEPS = None
    SCHEDULE_MAX_STEPS = None
    EVAL_STEPS = None
    OUTPUT_DIR = "/kaggle/working/checkpoints_heal_kl_50m_A"
elif SESSION.upper() == "B":
    PRESET = "heal_kl_50m"
    MAX_STEPS = 12207
    SAVE_OPTIMIZER = False
    RESUME = str(find_file("checkpoint-final"))
    SKIP_SHOCK = True
    SKIP_ORIG = True
    LEARNING_RATE = None
    LR_SCHEDULER = None
    MIN_LR_RATIO = None
    WARMUP_STEPS = None
    SCHEDULE_MAX_STEPS = None
    EVAL_STEPS = None
    OUTPUT_DIR = "/kaggle/working/checkpoints_heal_kl_50m_B"
    print("resume from:", RESUME)
elif SESSION.upper() == "P":
    # ~5M polish after KL-50M B; weights-only → new Adam; constant lr 2e-5
    PRESET = "polish_kl_5m"
    MAX_STEPS = None  # preset 13487
    SAVE_OPTIMIZER = False
    RESUME = str(find_file("checkpoint-final"))  # B weights-only OK
    SKIP_SHOCK = True
    SKIP_ORIG = True
    LEARNING_RATE = None  # preset 2e-5
    LR_SCHEDULER = None
    MIN_LR_RATIO = None
    WARMUP_STEPS = None
    SCHEDULE_MAX_STEPS = None  # preset 1280 (fresh sched horizon)
    EVAL_STEPS = None
    OUTPUT_DIR = "/kaggle/working/checkpoints_polish_kl_5m"
    print("polish resume from:", RESUME)
else:
    raise ValueError("SESSION must be 'A', 'B', or 'P'")

out = Path(OUTPUT_DIR)
if CLEAR_OUTPUT and out.exists():
    shutil.rmtree(out)
    print("cleared", out)
out.mkdir(parents=True, exist_ok=True)

ns = argparse.Namespace(
    preset=PRESET,
    model_name=None,
    train_data=str(train_path),
    val_data=str(val_path),
    output_dir=OUTPUT_DIR,
    seq_length=None,
    batch_size=None,
    max_steps=MAX_STEPS,
    max_eval_batches=20,
    max_train_texts=None,
    max_val_texts=None,
    skip_train=False,
    skip_shock=SKIP_SHOCK,
    skip_orig=SKIP_ORIG,
    resume=RESUME,
    no_bf16=False,
    no_8bit_adam=False,
    quant_warmup_steps=None,
    warmup_steps=WARMUP_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler=LR_SCHEDULER,
    min_lr_ratio=MIN_LR_RATIO,
    schedule_max_steps=SCHEDULE_MAX_STEPS,
    logging_steps=None,
    eval_steps=EVAL_STEPS,
    save_steps=None,
    save_optimizer=SAVE_OPTIMIZER,
    skip_linear_attn=None,
    no_skip_linear_attn=False,
    distill_alpha=None,
    distill_temperature=None,
    quant_reg_beta=None,
    seed=42,
    device_map="auto",
)
print(
    f"SESSION={SESSION} preset={PRESET} max_steps={MAX_STEPS} "
    f"save_optimizer={SAVE_OPTIMIZER} resume={RESUME} out={OUTPUT_DIR}"
)
print("note: KL loads frozen FP teacher (~2× VRAM)")
results = run_smoke(ns)
keys = [
    "preset", "ppl_original", "ppl_shock", "ppl_after_smoke",
    "loss_finite", "tokens_seen", "tokens_budget", "steps_ran",
    "resumed_step", "schedule_horizon_steps", "distill",
]
print({k: results[k] for k in keys if k in results})
if "inventory_summary" in results:
    inv = results["inventory_summary"]
    print("inventory", inv)
    if inv.get("n_eligible", 0) > 150:
        print("WARNING: eligible looks like all-Linear — GDN skip may be off")
ppl = results.get("ppl_after_smoke")
if ppl is not None:
    ref = results.get("ppl_original") or results.get("ppl_original_ref") or 17.67
    print(f"after/orig ≈ {ppl / ref:.3f} (ref orig {ref})")
    if SESSION.upper() == "A":
        print(f"Session A mid PPL={ppl:.2f} — go/no-go: continue B if ≲50–52 and falling")
        print("Upload OUTPUT_DIR checkpoint-final (FULL) as Kaggle Dataset for Session B")
    elif SESSION.upper() == "B":
        print(f"Session B final PPL={ppl:.2f} — bar: CE heal_50m ~43.77; strong if ≲35")
        print("Upload checkpoint-final for polish SESSION=P (weights-only OK)")
    else:
        print(f"Polish final PPL={ppl:.2f} — gate: < 34.38 = new SOTA; else stop polish")
        if ppl < 34.38:
            print("PASS — record in RESULTS.md")
        else:
            print("NO PASS — try α/T scout (§5.4) or FP CPT hygiene")

### Artifacts

**Session A** — full `checkpoint-final` (opt+sched) for B.  
**Session B** — weights-only final OK for polish P.  
**Session P** — `checkpoints_polish_kl_5m`: best/final + `metrics.jsonl`.

### Frozen baselines

| Run | ≈ tokens | Val PPL |
|-----|---------:|--------:|
| Original FP | — | ~17.7 |
| scout_kl_5m | 5.2M | ~49.3 |
| CE heal_50m | 50M | ~43.77 |
| **heal_kl_50m A+B** | **50M** | **~34.38** |
| polish_kl_5m | +5.2M | TBD (gate &lt;34.38) |

Record PPL in `RESULTS.md`.
